# How Underworld3 Turns SymPy into C

This notebook accompanies the blog post of the same name.
It walks through the pipeline from user-facing SymPy expressions
to compiled C callbacks registered with PETSc.

## At the User Level

We set up a Stokes flow problem with a temperature-dependent
(Frank-Kamenetskii) viscosity. This is a standard thermal convection
setup — simple enough to follow, complex enough to show the pipeline.

In [1]:
import underworld3 as uw
import sympy

### The mesh

In [2]:
mesh = uw.meshing.UnstructuredSimplexBox(
    minCoords=(0.0, 0.0),
    maxCoords=(1.0, 1.0),
    cellSize=0.1,
    qdegree=3,
)

x,y = mesh.X

### Parameters as UWexpressions

Each parameter is a symbolic name with a concrete value.
These behave like SymPy symbols in expressions but carry their values
for the compiler to extract later. (Units can be added via `uw.quantity()`
but we keep things dimensionless here to focus on the pipeline.)

In [3]:
# Parameters — symbolic names with concrete values (dimensionless here)
eta_0   = uw.expression(r"\eta_0", 1.0)    # reference viscosity
gamma   = uw.expression(r"\gamma", 13.8)   # FK sensitivity
Ra      = uw.expression("Ra", 1e6)         # Rayleigh number

### Mesh variables

Velocity, pressure, and temperature are mesh variables — fields
defined at every point in the mesh. We create them explicitly
so that they have readable names in the symbolic output.

In [4]:
v = uw.discretisation.MeshVariable(r"u", mesh, mesh.dim, degree=2)
p = uw.discretisation.MeshVariable(r"p", mesh, 1, degree=1, continuous=True)
T = uw.discretisation.MeshVariable(r"T", mesh, 1, degree=2)

### The Stokes solver

We pass our velocity and pressure variables to the solver.

In [5]:
stokes = uw.systems.Stokes(mesh, velocityField=v, pressureField=p)

### Building the constitutive law

Frank-Kamenetskii viscosity: $\eta = \eta_0 \exp(-\gamma T)$

This is ordinary SymPy arithmetic combining UWexpressions (parameters)
and a MeshVariable (spatial field). Nothing is evaluated yet.

In [6]:
viscosity_fn = eta_0 * sympy.exp(-gamma * T)
viscosity_fn

Matrix([[\eta_0*exp(-\gamma*{T}(N.x, N.y))]])

### Assigning the constitutive model and body force

In [7]:
stokes.constitutive_model = uw.constitutive_models.ViscousFlowModel
stokes.constitutive_model.Parameters.shear_viscosity_0 = viscosity_fn
stokes.bodyforce = sympy.Matrix([0, -Ra * T[0, 0]])

### Boundary conditions — free-slip on all walls

In [8]:
stokes.add_dirichlet_bc((sympy.oo, 0.0), "Top")
stokes.add_dirichlet_bc((sympy.oo, 0.0), "Bottom")
stokes.add_dirichlet_bc((0.0, sympy.oo), "Left")
stokes.add_dirichlet_bc((0.0, sympy.oo), "Right")

## Stage 1: The Strong Form Template

The Stokes equation in strong form is:

$$-\nabla \cdot \underbrace{\boldsymbol{\sigma}}_{\mathbf{F_1}}
  - \underbrace{\mathbf{f}}_{F_0} = 0$$

The solver decomposes this into $\mathbf{F}_1$ (the stress flux — everything
under the divergence) and $F_0$ (the body force — everything else).
We can inspect each piece separately.

Note that `.sym` let's us see through to the inner symbolic representation of an object

### The body force ($F_0$)

In [9]:
stokes.F0.sym

Matrix([
[               0],
[Ra*{T}(N.x, N.y)]])

### The constitutive stress ($\mathbf{F}_1$)

The constitutive model defines the stress as a function of the strain rate.
Our FK viscosity appears inside it. Note the presence of a penalty term that 
helps enforce incompressibility. 

In [10]:
stokes.F1.sym

Matrix([
[2*\eta*{u}_{ 0,0}(N.x, N.y) + \uplambda*({u}_{ 0,0}(N.x, N.y) + {u}_{ 1,1}(N.x, N.y)) - {p}(N.x, N.y),                                              2*\eta*({u}_{ 0,1}(N.x, N.y)/2 + {u}_{ 1,0}(N.x, N.y)/2)],
[                                             2*\eta*({u}_{ 0,1}(N.x, N.y)/2 + {u}_{ 1,0}(N.x, N.y)/2), 2*\eta*{u}_{ 1,1}(N.x, N.y) + \uplambda*({u}_{ 0,0}(N.x, N.y) + {u}_{ 1,1}(N.x, N.y)) - {p}(N.x, N.y)]])

### The viscosity parameter inside the model

In [11]:
stokes.constitutive_model.Parameters.shear_viscosity_0.sym

Matrix([[\eta_0*exp(-\gamma*{T}(N.x, N.y))]])

In [12]:
stokes.constitutive_model.Parameters.shear_viscosity_0.sym.diff(x)

Matrix([[-\eta_0*\gamma*{T}_{,0}(N.x, N.y)*exp(-\gamma*{T}(N.x, N.y))]])

These are all live SymPy expressions. The solver has not evaluated anything —
it stores them symbolically and defers compilation until `solve()` is called.
Note that the constitutive flux contains the strain rate of the unknown
velocity field, and the FK viscosity with its dependence on temperature.

## Stage 2: Automatic Jacobians

The solver differentiates $F_0$ and $\mathbf{F}_1$ with respect to the unknowns
to produce four Jacobian blocks $G_0$–$G_3$ for PETSc's Newton solver.

We can see what this means by differentiating the viscosity ourselves.
The viscosity depends on $T$, so any Jacobian term involving
$\partial\mathbf{F}_1/\partial T$ will contain this derivative:

In [13]:
sympy.diff(viscosity_fn, T)

[[[[-\eta_0*\gamma*exp(-\gamma*{T}(N.x, N.y))]]]]

SymPy computes exact symbolic derivatives. No finite-difference
approximations, no hand-coding.

## The Symbolic Wrappers

Let's look at what the expressions are actually made of.

### UWexpression: symbol with a value

In [14]:
# eta_0 is a SymPy symbol...
print(f"Type: {type(eta_0)}")
print(f"In an expression: {2 * eta_0 * gamma}")

# ...but it carries a value
print(f"Stored value: {eta_0.value}")

Type: <class 'underworld3.function.expressions.UWexpression'>
In an expression: 2*\eta_0*\gamma
Stored value: 1.00000000000000


In [15]:
# Display it — renders as LaTeX in the notebook
eta_0

<IPython.core.display.Latex object>

### MeshVariable symbols: spatial field data

In [16]:
# The temperature variable's symbolic face
T.sym

Matrix([[{T}(N.x, N.y)]])

In [17]:
# The velocity variable — a vector of symbols
v.sym

Matrix([[{u}_{ 0 }(N.x, N.y), {u}_{ 1 }(N.x, N.y)]])

### Coordinates: also symbolic

In [18]:
# Mesh coordinates as SymPy symbols
mesh.X

In [19]:
# Use them in expressions — depth-dependent body force
depth_force = Ra * mesh.X[1]
depth_force

N.y*Ra

### UWexpression values

In [20]:
# Each expression carries its current value
print(f"eta_0 = {eta_0.value}")
print(f"gamma = {gamma.value}")
print(f"Ra    = {Ra.value}")

eta_0 = 1.00000000000000
gamma = 13.8000000000000
Ra    = 1000000.00000000


## Summary

Everything you see in this notebook — the parameters, the viscosity law,
the $F_0$/$\mathbf{F}_1$ terms, the constitutive flux — is simultaneously:

- **Human-readable mathematics** (rendered in the notebook)
- **A complete specification for C code generation** (the JIT compiler reads it)
- **Symbolically differentiable** (for automatic Jacobians)

The compiler just reads what was there all along.